In [17]:
!pip install googletrans==3.1.0a0
!pip install langdetect


In [18]:
!pip install rapidfuzz

In [2]:
!pip install docx

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for docx: filename=docx-0.2.4-py3-none-any.whl size=53903 sha256=48f6ce68b560055d9c22e59cee4d6f0999a8e8523b5a4b345533e1f80e792070
  Stored in directory: c:\users\zakaria\appdata\local\pip\cache\wheels\f3\ba\dd\43ed5f165600f41deddeb1e382c56ffc1067c09ec5bd705f39
Successfully built docx


In [5]:
import nltk
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')


try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
import numpy as np
import pandas as pd
from langdetect import detect
from googletrans import Translator
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from collections import Counter
# from docx import Document
import os
# from google.colab import drive

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Zakaria\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# **Read jop descriptions in dataframe**

This step introduces the raw job description data into the pipeline.
The dataset represents real-world job postings collected from LinkedIn and serves as the primary textual source for all subsequent preprocessing and skill extraction steps.
At this stage, the data remains unstructured and is not yet suitable for modeling.


In [6]:
first_df = pd.read_csv(r"/content/linkedin_jobs_egypt.csv")
first_df


FileNotFoundError: [Errno 2] No such file or directory: '/content/linkedin_jobs_egypt.csv'

In [21]:
second_df = pd.read_csv(r"/content/linkedin_jobs_egypt_combined.csv")
second_df

,url,job_title,company,company_url,location,job_type,job_function,career_level,salary,posted_date,industries,summary,responsibilities,qualifications
0,https://www.linkedin.com/jobs/view/office-admi...,Office Administrator,Fawry MSME Finance,https://eg.linkedin.com/company/fawry-msme-fin...,"Cairo, Cairo, Egypt",Full-time,Administrative,Executive,NaN,1 day ago,Financial Services,Oversee daily administrative operations relate...,NaN,"Maintain records, contracts, and documentation..."
1,https://www.linkedin.com/jobs/view/data-entry-...,Data Entry Clerk,Aujan Coca-Cola Beverages Company (ACCBC),https://ae.linkedin.com/company/aujan-industri...,"Giza, Al Jizah, Egypt",Contract,Administrative,Entry level,NaN,2 weeks ago,Food and Beverage Manufacturing,"Job Description:\nThe Data Entry Clerk, under ...","Accurately record and process vendor invoices,...",Record and process employee expense claims in ...
2,https://www.linkedin.com/jobs/view/cabin-crew-...,Cabin Crew - Cairo,Air Arabia,https://ae.linkedin.com/company/air-arabia?trk...,"Cairo, Cairo, Egypt",Full-time,Management and Manufacturing,Entry level,NaN,2 weeks ago,Airlines and Aviation,Job Purpose\nActs as the airline’s ambassador;...,"Ensures timely attendance, proper grooming, fi...",Diploma or Higher Secondary Certificate is acc...
3,https://www.linkedin.com/jobs/view/accountant-...,Accountant,Fawry MSME Finance,https://eg.linkedin.com/company/fawry-msme-fin...,"Cairo, Cairo, Egypt",Full-time,Human Resources,Entry level,"EGP 12,000.00 - EGP 20,000.00",2 weeks ago,Financial Services,· Following Up on Insurance Default Cases & Ca...,NaN,"Bachelor’s degree in Accounting, Finance, or B..."
4,https://www.linkedin.com/jobs/view/community-m...,Community Manager,Wadi Degla Developments,https://eg.linkedin.com/company/wadi-degla-dev...,"Cairo, Egypt",Full-time,Customer Service,Mid-Senior level,NaN,3 days ago,Real Estate,Are you passionate about creating vibrant and ...,Resident & Community Engagement:\nAct as the m...,Education:\nBachelor’s degree in Business Admi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282,https://www.linkedin.com/jobs/view/striker-ale...,Striker - Alexandria,Red Bull,https://at.linkedin.com/company/red-bull?trk=p...,"Alexandria, Alexandria, Egypt",Full-time,Sales,Entry level,NaN,1 week ago,Food and Beverage Services,"About the Role:\nAs a Striker, you are Red Bul...",Territory Coverage\nEffective coverage of all ...,A solid understanding of the Off Premise chann...
283,https://www.linkedin.com/jobs/view/consultant-...,Consultant - Sales Management - Cairo Airport ...,Chalhoub Group,https://ae.linkedin.com/company/chalhoubgroup?...,"Cairo, Cairo, Egypt",Full-time,Business Development and Sales,Entry level,NaN,4 days ago,Retail Luxury Goods and Jewelry,INSPIRE | EXHILARATE | DELIGHT\nFor over six d...,The Sales Associate is responsible for Strengt...,NaN
284,https://www.linkedin.com/jobs/view/sales-promo...,Sales Promoter - Max Factor & Rimmel,Chalhoub Group,https://ae.linkedin.com/company/chalhoubgroup?...,"Cairo, Cairo, Egypt",Full-time,Sales and Business Development,Entry level,NaN,2 weeks ago,Retail Luxury Goods and Jewelry,INSPIRE | EXHILARATE | DELIGHT\nFor over six d...,Our Sales Promoter is responsible for deliveri...,NaN
285,https://www.linkedin.com/jobs/view/sales-staff...,Sales Staff,Palma,https://eg.linkedin.com/company/getpalma?trk=p...,"Cairo, Egypt",Full-time,Sales and Business Development,Entry level,NaN,1 day ago,Retail Apparel and Fashion,"🟩 We're hiring! – Sales Associates\nPalma, a l...",NaN,• Retail sales experience (fashion preferred)\...


In [24]:
df = df_all = pd.concat([first_df, second_df], ignore_index=True)
df

,url,job_title,company,company_url,location,job_type,job_function,career_level,salary,posted_date,industries,summary,responsibilities,qualifications
0,https://www.linkedin.com/jobs/view/cairo-cabin...,"Cairo Cabin Crew Opportunities (Dubai Based, R...",Emirates,https://ae.linkedin.com/company/emirates?trk=p...,"Cairo, Egypt",Full-time,"Customer Service, General Business, and Other",Associate,"Basic salary = AED 4,980 / month,",2 weeks ago,Airlines and Aviation and Hospitality,"Job Purpose\nA personality that shines, the ab...",NaN,Here are some other things we look for in our ...
1,https://www.linkedin.com/jobs/view/receptionis...,Receptionist,Al Ahly Momkn,https://eg.linkedin.com/company/alahlymomknfor...,"Qesm El Maadi, Cairo, Egypt",Full-time,Information Technology,Entry level,NaN,1 week ago,Financial Services,"Welcome guests and candidates, answering scree...",NaN,Not less than 3 years of previous experience a...
2,https://www.linkedin.com/jobs/view/future-lead...,Future Leaders Program 2025 - Marketing,Beyti - an Almarai Subsidiary,https://eg.linkedin.com/company/beyti-an-almar...,"Cairo, Egypt",Full-time,Marketing,Entry level,NaN,4 days ago,Food and Beverage Manufacturing,Beyti is looking for young graduates and early...,NaN,Participants benefit from a rich variety of st...
3,https://www.linkedin.com/jobs/view/account-man...,Account Manager - Key Accounts,Juhayna Food Industries,https://eg.linkedin.com/company/juhayna-food-i...,"Giza, Al Jizah, Egypt",Full-time,Sales and Distribution,Entry level,NaN,3 weeks ago,Food and Beverage Services,Job Purpose\nResponsible for managing relation...,NaN,"Bachelor's degree in Business Administration, ..."
4,https://www.linkedin.com/jobs/view/area-sales-...,Area Sales Manager,Juhayna Food Industries,https://eg.linkedin.com/company/juhayna-food-i...,"Sawhāj, Suhaj, Egypt",Full-time,Sales,Mid-Senior level,NaN,6 days ago,"Manufacturing, Dairy Product Manufacturing, an...",Job description:\nWho we are\nFor over three d...,"Convert the Region’s sales, net revenue, and m...",Responsible for communicating volume forecasts...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,https://www.linkedin.com/jobs/view/striker-ale...,Striker - Alexandria,Red Bull,https://at.linkedin.com/company/red-bull?trk=p...,"Alexandria, Alexandria, Egypt",Full-time,Sales,Entry level,NaN,1 week ago,Food and Beverage Services,"About the Role:\nAs a Striker, you are Red Bul...",Territory Coverage\nEffective coverage of all ...,A solid understanding of the Off Premise chann...
343,https://www.linkedin.com/jobs/view/consultant-...,Consultant - Sales Management - Cairo Airport ...,Chalhoub Group,https://ae.linkedin.com/company/chalhoubgroup?...,"Cairo, Cairo, Egypt",Full-time,Business Development and Sales,Entry level,NaN,4 days ago,Retail Luxury Goods and Jewelry,INSPIRE | EXHILARATE | DELIGHT\nFor over six d...,The Sales Associate is responsible for Strengt...,NaN
344,https://www.linkedin.com/jobs/view/sales-promo...,Sales Promoter - Max Factor & Rimmel,Chalhoub Group,https://ae.linkedin.com/company/chalhoubgroup?...,"Cairo, Cairo, Egypt",Full-time,Sales and Business Development,Entry level,NaN,2 weeks ago,Retail Luxury Goods and Jewelry,INSPIRE | EXHILARATE | DELIGHT\nFor over six d...,Our Sales Promoter is responsible for deliveri...,NaN
345,https://www.linkedin.com/jobs/view/sales-staff...,Sales Staff,Palma,https://eg.linkedin.com/company/getpalma?trk=p...,"Cairo, Egypt",Full-time,Sales and Business Development,Entry level,NaN,1 day ago,Retail Apparel and Fashion,"🟩 We're hiring! – Sales Associates\nPalma, a l...",NaN,• Retail sales experience (fashion preferred)\...


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   url               347 non-null    object
 1   job_title         347 non-null    object
 2   company           347 non-null    object
 3   company_url       347 non-null    object
 4   location          347 non-null    object
 5   job_type          347 non-null    object
 6   job_function      347 non-null    object
 7   career_level      347 non-null    object
 8   salary            55 non-null     object
 9   posted_date       347 non-null    object
 10  industries        347 non-null    object
 11  summary           310 non-null    object
 12  responsibilities  191 non-null    object
 13  qualifications    294 non-null    object
dtypes: object(14)
memory usage: 38.1+ KB


## **Text Preprocessing**

**1-Translation Step: Converting Job Descriptions to English**

The LinkedIn dataset contains job descriptions written in multiple languages (e.g., English, Arabic)
To ensure consistent text processing and skill extraction, all text fields are translated into English.




In [26]:
translator = Translator()

def translate_to_en(text):
    if pd.isna(text):
        return text
    try:
        lang = detect(text)
        if lang != 'en':
            translated = translator.translate(text, src='auto', dest='en')
            return translated.text
        return text
    except:
        return text

df_trans = df.copy(deep=True)
df_trans['summary'] = df_trans['summary'].apply(translate_to_en)
df_trans['qualifications'] = df_trans['qualifications'].apply(translate_to_en)
df_trans['responsibilities'] = df_trans['responsibilities'].apply(translate_to_en)

In [27]:
df_trans['qualifications'][1]

'Not less than 3 years of previous experience as a receptionist.\nGood Communication.\nOrganizer.\nMultitasking.\nGood Command of English language.\nProblem solving.\nBenefits\nEmbark on an exciting journey with the Fin-Tech Experts.\nJoin a workplace that actively encourages and supports all talents.\nA support system where you have a safe place to voice your opinion, share feedback, and be your true authentic self.\nJoin us in our mission to accelerate financial inclusion and make financial freedom accessible to all.'

In [28]:
df_trans['responsibilities'][20]

'Support onboarding, training, and employee engagement activities.\nAssist with recruitment by screening and uploading CVs.\nPrepare HR documents, organize files, and support during audits.\nRespond to employee queries and ensure compliance with HR policies.'

In [29]:
text_cols = ['summary', 'responsibilities', 'qualifications']

df_trans[text_cols] = df_trans[text_cols].fillna("")
df_trans['job_description_full'] = (
   df_trans['summary'] + " " +
   df_trans['responsibilities'] + " " +
   df_trans['qualifications']
)

In [36]:
df_trans['job_description_full'][0]

'Job Purpose\nA personality that shines, the ability to adapt to any situation and make people feel at ease. These are a few of the qualities we’re looking for in our cabin crew.\nAs the face of Emirates, you’ll be the person customers turn to for help and direction when they fly with us, so you need to be friendly, observant and able to provide the right support.\nBeing a member of the cabin crew is so much more than a service role - safety is our highest priority. You’ll need to lead confidently and take control when it comes to managing aircraft services, security, and safety procedures. This comes from the world-class learning experience our crew receive at our state-of-the-art facility in Dubai.  Here are some other things we look for in our cabin crew:\nYou’ve had more than a year’s experience in hospitality/customer service\nYou have a positive attitude and the natural ability to provide excellent service in a team environment, dealing with people from many cultures\nYour minimu

# Text Cleaning Method – Step Summary

1- Handle Missing Values

If the input is not a valid text (e.g., NaN or numeric), it is replaced with an empty string.
This prevents errors and keeps the preprocessing pipeline stable.

2- Text Normalization (Lowercasing)

All characters are converted to lowercase.
This ensures that words like Python and python are treated as the same term.

3- Remove Special Characters

Unnecessary symbols and uncommon characters are removed while keeping letters, numbers, and basic punctuation.
This reduces noise without losing important information.

4- Remove URLs

Any web links are removed from the text.
URLs do not contribute to skill extraction or semantic understanding.

5️- Remove Mentions

Mentions such as @company are removed.
These are usually metadata artifacts and add no value to content analysis.

6- Remove Boilerplate and Irrelevant Phrases

Common repetitive phrases found in job postings (e.g., legal statements or platform-specific text) are removed.
This step focuses the text on actual job requirements and skills.

7- Tokenization

The cleaned text is split into individual words.
This prepares the text for further linguistic processing.

8- Stopword Removal

Frequently occurring words with little semantic meaning (e.g., the, is, and) are removed.
This helps highlight important terms such as skills and responsibilities.

9️- Lemmatization

Words are reduced to their base form (e.g., skills → skill, developers → developer).
This improves consistency and helps group similar words together.

In [37]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def clean_text(text):
  if not isinstance(text, str):  # If it's NaN or float
        text = ""
  text = text.lower()
  pat = r'[^a-zA-Z.,!?/:;\"\'\s]'
  text = re.sub(pat,' ',text)
  text = re.sub(r'https?:\S*', '', text)
  text = re.sub(r'@\S*', '', text)
  text = re.sub(r'\b(view on map|additional information|job number| marriott|jw marriott|le ridien|equal opportunity|employee happy|join portfolio)\b', '', text)

  tokens = text.split()
  cleaned_tokens = [
      lemmatizer.lemmatize(word) for word in tokens if word not in stop_words
  ]
  return " ".join(cleaned_tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [38]:
df_trans['summary'] = df_trans['summary']
df_trans['qualifications'] = df_trans['qualifications']
df_trans['qualifications_cleaned'] = df_trans['qualifications'].apply(clean_text)
df_trans['summary_cleaned'] = df_trans['summary'].apply(clean_text)
df_trans['responsibilities_cleaned'] = df_trans['responsibilities'].apply(clean_text)
df_trans['job_description_clean'] = (
    df_trans['summary_cleaned'] + " " +
    df_trans['responsibilities_cleaned'] + " " +
    df_trans['qualifications_cleaned']
)

In [39]:
df_trans[['job_description_clean','job_description_full']]

,job_description_clean,job_description_full
0,"job purpose personality shines, ability adapt ...","Job Purpose\nA personality that shines, the ab..."
1,"welcome guest candidates, answering screening ...","Welcome guests and candidates, answering scree..."
2,beyti looking young graduate early career prof...,Beyti is looking for young graduates and early...
3,job purpose responsible managing relationship ...,Job Purpose\nResponsible for managing relation...
4,"job description: three decades, juhayna food i...",Job description:\nWho we are\nFor over three d...
...,...,...
342,"role: striker, red bull's face market. respons...","About the Role:\nAs a Striker, you are Red Bul..."
343,"inspire exhilarate delight six decades, chalho...",INSPIRE | EXHILARATE | DELIGHT\nFor over six d...
344,"inspire exhilarate delight six decades, chalho...",INSPIRE | EXHILARATE | DELIGHT\nFor over six d...
345,"hiring! sale associate palma, leading fashion ...","🟩 We're hiring! – Sales Associates\nPalma, a l..."


## Skill Extraction & Feature Engineering

This code converts raw job description text into structured, machine-learning-ready features using a **rule-based exact matching approach**.

### Skill Dictionaries
Two curated dictionaries are defined:
- **Hard Skills Dictionary**: Technical, engineering, data, business, and platform-related skills.
- **Soft Skills Dictionary**: Behavioral, communication, leadership, and interpersonal skills.

Each dictionary maps:
- A **canonical skill name** (used as a feature)
- To multiple **textual variants** that may appear in job descriptions

This normalizes different surface forms into a single skill representation.

---

###  Skill Extraction
For each cleaned job description:
- The text is scanned for predefined skill variants
- If any variant is found, the corresponding canonical skill is added
- Each skill is counted **once per job description**

This produces two new columns:
- `required_hard_skills`
- `required_soft_skills`

---

###  Feature Construction
From the extracted skills:
- Skills can be encoded as **binary presence features**
- A numerical feature `skill_count` is created by summing hard and soft skills

This feature captures the **breadth and complexity** of job requirements.

---

###  Outcome
The pipeline transforms unstructured job description text into:
- Structured hard-skill features
- Structured soft-skill features
- An aggregate skill-count signal

These features are directly usable by machine learning models and complement other engineered features.


In [54]:
HARD_SKILLS_DICT = {
    # Programming & Development
    "python": ["python"],
    "software development": ["software", "software development"],
    "backend development": ["backend", "backend development"],
    "web development": ["web", "web development"],
    "cloud computing": ["cloud"],
    "automation": ["automation"],
    "testing": ["testing"],
    "frameworks": ["frameworks"],
    "code": ["code", "coding"],

    # Data & AI
    "data analysis": ["data analysis", "analysis"],
    "data science": ["data science"],
    "machine learning": ["machine learning", "machine"],
    "ai": ["ai", "artificial intelligence"],
    "models": ["models", "model"],
    "reporting": ["reporting", "reports"],

    # Engineering
    "engineering": ["engineering"],
    "systems engineering": ["systems", "system"],
    "networking": ["network", "networking"],

    # Business & Finance
    "financial analysis": ["financial", "finance"],
    "accounting": ["accounting"],
    "sales": ["sales"],
    "marketing": ["marketing", "digital marketing"],
    "product management": ["product"],
    "operations management": ["operations"],
    "project management": ["project", "projects"],

    # Tools & Platforms
    "microsoft office": ["microsoft", "office"],
    "crm systems": ["crm"],
    "software tools": ["tools"],
    "applications": ["applications"],
    "technology platforms": ["technology", "technologies"],

    # HR & Recruitment
    "recruitment": ["recruitment", "hiring", "talent acquisition"],
    "human resources": ["employee", "candidates", "candidate"]
}

SOFT_SKILLS_DICT = {
    "communication": ["communication"],
    "teamwork": ["team", "teams", "collaborate", "collaboration"],
    "problem solving": ["problem", "issues", "solutions"],
    "leadership": ["lead", "manager", "management"],
    "time management": ["time", "timely"],
    "customer focus": ["customer", "customers", "client", "clients", "service"],
    "adaptability": ["learning", "innovation"],
    "attention to detail": ["quality", "standards", "compliance"],
    "organizational skills": ["organization", "process", "processes"],
    "analytical thinking": ["analysis", "identify", "understanding"],
    "work ethic": ["responsible", "accountable"],
    "collaboration": ["across", "internal", "external"],
    "growth mindset": ["growth", "improve", "develop"]
}

def extract_skills_from_text(text, skill_dict):
    found = []
    for canonical, variants in skill_dict.items():
        for v in variants:
            if v in text:
                found.append(canonical)
                break
    return found

def vectorize_skills(skill_list, master_skills):
    return [1 if s in skill_list else 0 for s in master_skills]

In [55]:
MASTER_HARD_SKILLS = list(HARD_SKILLS_DICT.keys())
MASTER_SOFT_SKILLS = list(SOFT_SKILLS_DICT.keys())

In [56]:
df_trans['required_hard_skills'] = df_trans['job_description_clean'].apply(
    lambda x: extract_skills_from_text(x, HARD_SKILLS_DICT)
)


df_trans['required_soft_skills'] = df_trans['job_description_clean'].apply(
    lambda x: extract_skills_from_text(x, SOFT_SKILLS_DICT)
)


df_trans['skill_count'] = (
    df_trans['required_hard_skills'].apply(len) +
    df_trans['required_soft_skills'].apply(len)
)

In [57]:
df_trans[['job_function', 'required_hard_skills', 'required_soft_skills', 'skill_count','job_description_clean']].head(50)

,job_function,required_hard_skills,required_soft_skills,skill_count,job_description_clean
0,"Customer Service, General Business, and Other",[ai],"[teamwork, leadership, customer focus, adaptab...",6,"job purpose personality shines, ability adapt ..."
1,Information Technology,"[ai, systems engineering, financial analysis, ...","[communication, problem solving]",6,"welcome guest candidates, answering screening ..."
2,Marketing,"[ai, marketing, microsoft office]","[teamwork, leadership, growth mindset]",6,beyti looking young graduate early career prof...
3,Sales and Distribution,"[ai, sales, product management]","[leadership, time management, customer focus, ...",8,job purpose responsible managing relationship ...
4,Sales,"[ai, models, networking, sales, product manage...","[teamwork, problem solving, leadership, custom...",16,"job description: three decades, juhayna food i..."
5,Finance,"[data analysis, ai, models, engineering, syste...","[teamwork, leadership, time management, organi...",16,manage standard costing process manufactured ...
6,Engineering,"[ai, reporting, systems engineering, financial...","[teamwork, problem solving, leadership, time m...",19,"subsidiary al ahly capital, al ahly momkn fast..."
7,Finance and Accounting/Auditing,"[ai, reporting, networking, financial analysis...","[teamwork, problem solving, leadership, time m...",17,"three decades, juhayna food industry pioneer e..."
8,Business Development,"[systems engineering, financial analysis]","[leadership, customer focus, growth mindset]",5,sale account manager criterion condition busin...
9,"Administrative, General Business, and Management","[ai, reporting, engineering, networking, finan...","[teamwork, leadership, time management, custom...",17,"job description: three decades, juhayna food i..."


In [58]:
for i in df_trans[df_trans['job_function'] == 'Information Technology']['required_hard_skills']: print(i)

['ai', 'systems engineering', 'financial analysis', 'human resources']
['software development', 'testing', 'ai', 'reporting', 'product management', 'project management']
['software development', 'ai', 'systems engineering', 'networking', 'financial analysis', 'operations management', 'project management', 'microsoft office', 'technology platforms', 'human resources']
['testing', 'code', 'ai', 'engineering', 'systems engineering']
['backend development', 'cloud computing', 'testing', 'code', 'ai', 'engineering', 'systems engineering', 'product management', 'technology platforms']
['software development', 'ai', 'reporting', 'systems engineering', 'networking', 'applications']
['software development', 'ai', 'models', 'networking', 'microsoft office', 'human resources']
['software development', 'ai', 'systems engineering', 'networking', 'microsoft office', 'technology platforms', 'recruitment']
['networking', 'project management']
['ai', 'engineering', 'networking', 'microsoft office', 'te

In [48]:
df_trans[['job_function','required_hard_skills', 'required_soft_skills', 'skill_count','job_description_clean','job_description_full']].to_csv("skills_data_clean_.csv", index=False, encoding="utf-8")

In [49]:
df_trans['job_function'].unique()

array(['Customer Service, General Business, and Other',
       'Information Technology', 'Marketing', 'Sales and Distribution',
       'Sales', 'Finance', 'Engineering',
       'Finance and Accounting/Auditing', 'Business Development',
       'Administrative, General Business, and Management',
       'Customer Service and Sales', 'Finance and Human Resources',
       'Design', 'Accounting/Auditing',
       'Design, Art/Creative, and Information Technology',
       'Finance, Analyst, and Strategy/Planning',
       'Sales and Business Development', 'Human Resources',
       'Administrative', 'Other',
       'Research, Strategy/Planning, and Project Management',
       'Customer Service',
       'Administrative, Customer Service, and Public Relations',
       'Purchasing, Sales, and Administrative',
       'Engineering and Information Technology',
       'Accounting/Auditing, Finance, and Other',
       'Management and Manufacturing', 'Supply Chain',
       'Engineering, Consulting, and C

In [50]:
df_trans[df_trans['job_function'] == 'Information Technology'][['job_description_clean']]

,job_description_clean
1,"welcome guest candidates, answering screening ..."
36,"quality assurance junior engineer, play vital ..."
41,"group ib: founded headquartered singapore, gro..."
67,design implement window o application using ...
70,job objective backend development key account...
77,technical support: provide advanced technical ...
78,: major functions: technical support pc access...
79,hiring desktop support engineer egypt! excis g...
81,plan network architecture within sphere operat...
82,hiring back office network engineer fresh grad...


In [51]:
df_trans[df_trans['job_function'] == 'Other']['job_description_clean']

,job_description_clean
23,"seeking experienced civil, electrical, mechani..."
25,"company description ""why work accor? far world..."
31,emirate nbd egypt proud introduce business ban...
35,"company description join u accor, life pulse p..."
48,: year experience collect payment past due bi...
75,valleysoft center excellence regional service ...
76,"support professional remote ller's solutions, ..."
100,tagaddod forefront promoting sustainable waste...
101,tagaddod forefront promoting sustainable waste...
106,"job summary reporting engineering manager, aut..."


In [52]:
df_trans[['job_description_clean','required_hard_skills','required_soft_skills','skill_count']]

,job_description_clean,required_hard_skills,required_soft_skills,skill_count
0,"job purpose personality shines, ability adapt ...",[ai],"[teamwork, leadership, customer focus, adaptab...",6
1,"welcome guest candidates, answering screening ...","[ai, systems engineering, financial analysis, ...","[communication, problem solving]",6
2,beyti looking young graduate early career prof...,"[ai, marketing, microsoft office]","[teamwork, leadership, growth mindset]",6
3,job purpose responsible managing relationship ...,"[ai, sales, product management]","[leadership, time management, customer focus, ...",8
4,"job description: three decades, juhayna food i...","[ai, models, networking, sales, product manage...","[teamwork, problem solving, leadership, custom...",16
...,...,...,...,...
342,"role: striker, red bull's face market. respons...","[ai, reporting, product management, microsoft ...","[communication, teamwork, leadership, time man...",12
343,"inspire exhilarate delight six decades, chalho...","[ai, marketing, product management, human reso...","[teamwork, problem solving, leadership, custom...",13
344,"inspire exhilarate delight six decades, chalho...","[ai, sales, marketing, product management, hum...","[leadership, time management, customer focus, ...",12
345,"hiring! sale associate palma, leading fashion ...","[ai, recruitment]","[communication, teamwork, leadership, customer...",6


In [65]:
merged = df_trans.copy()

In [60]:
merged = merged.drop_duplicates(subset=["job_title", "company", "location"], keep="first")


In [66]:
merged['domain'] = merged['job_function'].str.lower().str.strip()

In [67]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   url                       347 non-null    object
 1   job_title                 347 non-null    object
 2   company                   347 non-null    object
 3   company_url               347 non-null    object
 4   location                  347 non-null    object
 5   job_type                  347 non-null    object
 6   job_function              347 non-null    object
 7   career_level              347 non-null    object
 8   salary                    55 non-null     object
 9   posted_date               347 non-null    object
 10  industries                347 non-null    object
 11  summary                   347 non-null    object
 12  responsibilities          347 non-null    object
 13  qualifications            347 non-null    object
 14  job_description_full      

In [68]:
arr = merged["job_function"].drop_duplicates().values
print(len(arr))

65


In [69]:
# Mapping from job function components to domains
domain_map = {
    "customer service": "Support",
    "information technology": "IT",
    "marketing": "Business",
    "sales": "Business",
    "sales and distribution": "Business",
    "finance": "Finance",
    "finance and accounting/auditing": "Finance",
    "engineering": "Engineering",
    "business development": "Business",
    "administrative": "Administrative",
    "human resources": "HR",
    "design": "Creative",
    "research": "Research",
    "project management": "Management",
    "consulting": "Consulting",
    "supply chain": "Operations",
    "production": "Operations",
    "quality assurance": "Engineering",
    "analyst": "Analytics",
    "other": "Other",
    "advertising": "Business",
    "strategy/planning": "Management",
    "product management": "Management",
    "purchasing": "Operations",
    "writing/editing": "Creative"
}

def map_job_function_to_single_domain(job_func):
    # Lowercase and clean
    job_func = str(job_func).lower().strip()

    # Split by common separators
    parts = [p.strip() for p in job_func.replace("/", ",").replace("&", ",").replace(" and ", ",").split(",")]

    # Return first valid mapped domain
    for part in parts:
        if part in domain_map:
            return domain_map[part]
    # fallback if none matched
    return "Other"

# Apply to dataframe
merged['domain'] = merged['job_function'].apply(map_job_function_to_single_domain)

# Check result
print(merged[['job_function', 'domain']].head(20))
print("Unique domains:", merged['domain'].unique())


                                        job_function          domain
0      Customer Service, General Business, and Other         Support
1                             Information Technology              IT
2                                          Marketing        Business
3                             Sales and Distribution        Business
4                                              Sales        Business
5                                            Finance         Finance
6                                        Engineering     Engineering
7                    Finance and Accounting/Auditing         Finance
8                               Business Development        Business
9   Administrative, General Business, and Management  Administrative
10                        Customer Service and Sales         Support
11                       Finance and Human Resources         Finance
12                                             Sales        Business
13                                

In [70]:
merged["domain"].value_counts()

,count
domain,
Engineering,72
Other,48
IT,47
Business,43
HR,42
Finance,34
Administrative,24
Support,12
Research,8


In [71]:
arr = merged["domain"].drop_duplicates().values
print(len(arr))

14


In [72]:
merged.head()

,url,job_title,company,company_url,location,job_type,job_function,career_level,salary,posted_date,...,qualifications,job_description_full,qualifications_cleaned,summary_cleaned,responsibilities_cleaned,job_description_clean,required_hard_skills,required_soft_skills,skill_count,domain
0,https://www.linkedin.com/jobs/view/cairo-cabin...,"Cairo Cabin Crew Opportunities (Dubai Based, R...",Emirates,https://ae.linkedin.com/company/emirates?trk=p...,"Cairo, Egypt",Full-time,"Customer Service, General Business, and Other",Associate,"Basic salary = AED 4,980 / month,",2 weeks ago,...,Here are some other things we look for in our ...,"Job Purpose\nA personality that shines, the ab...",thing look cabin crew: year experience hospita...,"job purpose personality shines, ability adapt ...",,"job purpose personality shines, ability adapt ...",[ai],"[teamwork, leadership, customer focus, adaptab...",6,Support
1,https://www.linkedin.com/jobs/view/receptionis...,Receptionist,Al Ahly Momkn,https://eg.linkedin.com/company/alahlymomknfor...,"Qesm El Maadi, Cairo, Egypt",Full-time,Information Technology,Entry level,NaN,1 week ago,...,Not less than 3 years of previous experience a...,"Welcome guests and candidates, answering scree...",less year previous experience receptionist. go...,"welcome guest candidates, answering screening ...",,"welcome guest candidates, answering screening ...","[ai, systems engineering, financial analysis, ...","[communication, problem solving]",6,IT
2,https://www.linkedin.com/jobs/view/future-lead...,Future Leaders Program 2025 - Marketing,Beyti - an Almarai Subsidiary,https://eg.linkedin.com/company/beyti-an-almar...,"Cairo, Egypt",Full-time,Marketing,Entry level,NaN,4 days ago,...,Participants benefit from a rich variety of st...,Beyti is looking for young graduates and early...,participant benefit rich variety structured tr...,beyti looking young graduate early career prof...,,beyti looking young graduate early career prof...,"[ai, marketing, microsoft office]","[teamwork, leadership, growth mindset]",6,Business
3,https://www.linkedin.com/jobs/view/account-man...,Account Manager - Key Accounts,Juhayna Food Industries,https://eg.linkedin.com/company/juhayna-food-i...,"Giza, Al Jizah, Egypt",Full-time,Sales and Distribution,Entry level,NaN,3 weeks ago,...,"Bachelor's degree in Business Administration, ...",Job Purpose\nResponsible for managing relation...,"bachelor's degree business administration, rel...",job purpose responsible managing relationship ...,,job purpose responsible managing relationship ...,"[ai, sales, product management]","[leadership, time management, customer focus, ...",8,Business
4,https://www.linkedin.com/jobs/view/area-sales-...,Area Sales Manager,Juhayna Food Industries,https://eg.linkedin.com/company/juhayna-food-i...,"Sawhāj, Suhaj, Egypt",Full-time,Sales,Mid-Senior level,NaN,6 days ago,...,Responsible for communicating volume forecasts...,Job description:\nWho we are\nFor over three d...,responsible communicating volume forecast lead...,"job description: three decades, juhayna food i...","convert region sales, net revenue, market deve...","job description: three decades, juhayna food i...","[ai, models, networking, sales, product manage...","[teamwork, problem solving, leadership, custom...",16,Business


In [73]:
# Show all job functions that are mapped to "Other"
other_jobs = merged[merged['domain'] == "Other"]['job_function'].unique()
print("Job functions currently labeled as 'Other':")
for job in other_jobs:
    print("-", job)


Job functions currently labeled as 'Other':
- Accounting/Auditing
- Other
- Management and Manufacturing
- Management
- General Business, Management, and Other
- General Business and Accounting/Auditing
- General Business


In [74]:
manual_domain_map = {
    "Accounting/Auditing": "Finance",
    "Other": "Other",  # You can keep this as fallback
    "Management and Manufacturing": "Management",
    "Management": "Management",
    "General Business, Management, and Other": "Business",
    "General Business and Accounting/Auditing": "Business",
    "General Business": "Business"
}
merged['domain'] = merged.apply(
    lambda row: manual_domain_map.get(row['job_function'], row['domain']),
    axis=1
)
print("Unique domains after manual mapping:", merged['domain'].unique())
print(merged[merged['domain'] == "Other"]['job_function'].unique())  # Should be empty or very few


Unique domains after manual mapping: ['Support' 'IT' 'Business' 'Finance' 'Engineering' 'Administrative'
 'Creative' 'HR' 'Other' 'Research' 'Operations' 'Management' 'Consulting'
 'Analytics']
['Other']


In [75]:
merged["domain"].value_counts()

,count
domain,
Engineering,72
IT,47
Business,46
HR,42
Finance,39
Other,38
Administrative,24
Support,12
Research,8


In [76]:
# See all job functions that are still "Other"
remaining_other = merged[merged['domain'] == "Other"]['job_function'].unique()
print("Remaining job functions still labeled as 'Other':")
for job in remaining_other:
    print("-", job)


Remaining job functions still labeled as 'Other':
- Other


In [77]:
# Get all rows where domain is "Other"
other_jobs_df = merged[merged['domain'] == "Other"]

# Show relevant columns for inspection
display_cols = ['job_title', 'company', 'location', 'job_function']
print(other_jobs_df[display_cols])


                                             job_title  \
23                               Engineering Positions   
25                     Guest Relation Officer - Female   
31             Relationship Officer - Business Banking   
35     Guest Relations Officer - Sofitel Nile Downtown   
48                          Collection Officer Officer   
75         1st Level IT Support - 2nd Level IT Support   
76                                 IT Support - Remote   
100                          Sustainability Specialist   
101                   Senior Sustainability Specialist   
106                                Automation Engineer   
107         Frontend Engineer I/II/III/Staff/Sr. Staff   
110                       Front End Developer (Remote)   
111                                      Web Developer   
113                        Software Developer (Nodejs)   
125                            Data Science Specialist   
126                                     Data Scientist   
128           

# Final Dataset

In [82]:
drive.mount('/content/drive')
cv_folder = "/content/drive/MyDrive/job_recommendation_dataset/cvs"

cv_docs = []
for i, file in enumerate(os.listdir(cv_folder), 1):
    if file.endswith(".docx"):
        doc = Document(os.path.join(cv_folder, file))
        text = " ".join([p.text for p in doc.paragraphs])
        cv_docs.append({"cv_id": i, "text": text})

cv_df = pd.DataFrame(cv_docs)

# preprocessing
cv_df["clean_text"] = cv_df["text"].apply(clean_text)

# skill extraction
cv_df["hard_skills"] = cv_df["clean_text"].apply(
    lambda x: extract_skills_from_text(x, HARD_SKILLS_DICT)
)

cv_df["soft_skills"] = cv_df["clean_text"].apply(
    lambda x: extract_skills_from_text(x, SOFT_SKILLS_DICT)
)

# vectorization
cv_df["hard_vector"] = cv_df["hard_skills"].apply(
    lambda x: vectorize_skills(x, MASTER_HARD_SKILLS)
)

cv_df["soft_vector"] = cv_df["soft_skills"].apply(
    lambda x: vectorize_skills(x, MASTER_SOFT_SKILLS)
)


NameError: name 'drive' is not defined

## 🔗 CV–Job Matching Feature Computation

This function computes structured matching features between a candidate CV and a job description based on extracted hard and soft skill vectors.

### 🔹 Inputs
- `cv_hard_vec`: Binary vector representing hard skills present in the CV  
- `jd_hard_vec`: Binary vector representing hard skills required by the job  
- `cv_soft_vec`: Binary vector representing soft skills present in the CV  
- `jd_soft_vec`: Binary vector representing soft skills required by the job  

All vectors are aligned with predefined master skill lists.

---

###  Computed Features
The function returns the following matching signals:

- **`matched_weight`**:  
  Number of required hard skills that are present in the CV.

- **`required_weight`**:  
  Total number of hard skills required by the job.

- **`match_ratio`**:  
  Proportion of required hard skills matched by the CV.

- **`missing_required_skills_count`**:  
  Number of required hard skills missing from the CV.

- **`missing_hard_skills`**:  
  List of required hard skills that are not present in the CV.

- **`missing_soft_skills`**:  
  List of required soft skills that are not present in the CV.

---

###  Purpose
These features quantify the **degree of alignment** between a CV and a job description by measuring:
- Skill coverage
- Missing requirements
- Relative match quality

They provide interpretable, model-ready signals for downstream ranking or classification models.


In [83]:
def compute_match_features(cv_hard_vec, jd_hard_vec, cv_soft_vec, jd_soft_vec):
    matched_weight = sum([1 for i in range(len(cv_hard_vec)) if cv_hard_vec[i] and jd_hard_vec[i]])
    required_weight = sum(jd_hard_vec)
    match_ratio = matched_weight / required_weight if required_weight else 0
    missing_required_skills_count = sum([1 for i in range(len(jd_hard_vec)) if jd_hard_vec[i] and not cv_hard_vec[i]])
    missing_hard_skills = [MASTER_HARD_SKILLS[i] for i in range(len(jd_hard_vec)) if jd_hard_vec[i] and not cv_hard_vec[i]]
    missing_soft_skills = [MASTER_SOFT_SKILLS[i] for i in range(len(jd_soft_vec)) if jd_soft_vec[i] and not cv_soft_vec[i]]
    return matched_weight, required_weight, match_ratio, missing_required_skills_count, missing_hard_skills, missing_soft_skills


## 🔄 CV–Job Pairwise Feature Generation

This step generates **pairwise matching features** between every CV and every job description by computing skill alignment metrics.

---

### 🔹 Pairwise Matching Logic
For each `(CV, Job)` pair:
- Hard and soft skill vectors are compared
- A set of matching and missing-skill features is computed using the `compute_match_features` function

This creates a structured representation of how well a candidate matches a specific job.

---

### 🔹 Generated Features
For each CV–JD pair, the following features are produced:

- `cv_id`: Identifier of the CV  
- `jd_id`: Identifier of the job description  

- `matched_weight`:  
  Number of required hard skills present in the CV

- `required_weight`:  
  Total number of hard skills required by the job

- `match_ratio`:  
  Fraction of required hard skills matched by the CV

- `missing_required_skills_count`:  
  Number of required hard skills missing from the CV

- `missing_hard_skills`:  
  List of required hard skills not found in the CV

- `missing_soft_skills`:  
  List of required soft skills not found in the CV

- `required_hard_skills`:  
  Hard skills required by the job

- `required_soft_skills`:  
  Soft skills required by the job

- `required_skill_count`:  
  Total number of required skills for the job (hard + soft)

---

### 🔹 Output
All CV–JD pairs are aggregated into a single DataFrame and saved as:



In [84]:
rows = []
for _, cv in cv_df.iterrows():
    for _, jd in jd_df.iterrows():
        mw, rw, mr, miss_count, miss_hard, miss_soft = compute_match_features(
            cv["hard_vector"], jd["hard_vector"], cv["soft_vector"], jd["soft_vector"]
        )
        rows.append({
            "cv_id": cv["cv_id"],
            "jd_id": jd["jd_id"],
            "matched_weight": mw,
            "required_weight": rw,
            "match_ratio": mr,
            "missing_required_skills_count": miss_count,
            "missing_hard_skills": miss_hard,
            "missing_soft_skills": miss_soft,
            "required_hard_skills": jd["required_hard_skills"],
            "required_soft_skills": jd["required_soft_skills"],
            "required_skill_count": len(jd["required_hard_skills"]) + len(jd["required_soft_skills"])
        })

final_features = pd.DataFrame(rows)
final_features.to_csv("cv_jd_matching_features.csv", index=False)

NameError: name 'cv_df' is not defined